# Experimental Finder API

This notebook introduces the developer-friendly experimental API in `TCT.experimental`. The functions wrap the lower-level Translator query boilerplate so you can start with labels like `"asthma"` or CURIEs like `"MONDO:0004979"`.

The API is experimental: import from `TCT.experimental`, pin behavior in your own notebooks or applications, and expect the interface to evolve as it graduates into the stable package surface.

In [ ]:
from TCT.experimental import (
    clear_translator_resource_cache,
    get_translator_resources,
    neighborhood_finder,
    pathfinder,
)

## Resource Cache

Translator API metadata is expensive to fetch. The experimental API loads it lazily on the first query and reuses it for later calls in the same Python process.

Use `get_translator_resources()` when you want to preload metadata once, pass it to multiple calls, or make caching explicit in a notebook. Use `clear_translator_resource_cache()` when you want the next call to refetch metadata.

In [ ]:
resources = get_translator_resources()
resources

## Pathfinder

`pathfinder(start, end, intermediate_categories)` searches for two-hop paths between two concepts.

Inputs:

- `start`: start concept as a display string or CURIE.
- `end`: end concept as a display string or CURIE.
- `intermediate_categories`: allowed categories for the connecting node. Short names such as `"Gene"` are automatically converted to `"biolink:Gene"`.

The return value is a `FinderResult` with `.knowledge_graph`, `.results`, `.auxiliary_graphs`, `.resolved_nodes`, `.raw`, and `.to_dict()`.

In [ ]:
paths = pathfinder(
    start="asthma",
    end="albuterol",
    intermediate_categories=["Gene", "Protein"],
    resources=resources,
)
paths.resolved_nodes

In [ ]:
len(paths.knowledge_graph.get("nodes", {})), len(paths.knowledge_graph.get("edges", {}))

## Neighborhood Finder

`neighborhood_finder(node, neighbor_categories)` searches for one-hop neighbors of a concept. `node` can be a single string/CURIE or a list of strings/CURIEs.

In [ ]:
neighbors = neighborhood_finder(
    node="asthma",
    neighbor_categories=["SmallMolecule", "Drug"],
    resources=resources,
)
neighbors.resolved_nodes

In [ ]:
neighbors.knowledge_graph.keys()

## CURIE Inputs

When an input contains `:` and no spaces, the experimental API treats it as a CURIE and skips Name Resolver. It still uses Node Normalizer to get the preferred identifier, label, and categories.

In [ ]:
curie_neighbors = neighborhood_finder(
    node="MONDO:0004979",
    neighbor_categories=["Gene"],
    resources=resources,
)
curie_neighbors.resolved_nodes["node"]

## Refreshing Metadata

Use `refresh=True` to refetch metadata, or clear the cache before the next query.

In [ ]:
fresh_resources = get_translator_resources(refresh=True)
clear_translator_resource_cache()